# 📉 Customer Churn Prediction Pipeline
**Author:** Oluwasegun Raphael  
**Tools:** Python, Pandas, Scikit-Learn, Statsmodels, Matplotlib, Seaborn  
**Dataset:** Telco Customer Churn (IBM/Kaggle)

---
### Business Context
A telecom company is losing customers to competitors. Acquiring a new customer costs 5x more than retaining an existing one.
This pipeline builds a machine learning model to predict which customers are likely to churn, enabling the business
to intervene proactively with targeted retention strategies before customers leave.

### Objectives
1. Identify key drivers of customer churn
2. Build and compare multiple ML models
3. Select the best model based on business-relevant metrics
4. Generate actionable retention recommendations


---
## STEP 1 — Import Libraries

In [ ]:
# Core
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, accuracy_score, precision_score, recall_score, f1_score
)

# Stats
import statsmodels.api as sm

# Save model
import joblib

print('✅ All libraries imported successfully')

---
## STEP 2 — Load & Explore the Dataset

In [ ]:
# Load dataset — make sure WA_Fn-UseC_-Telco-Customer-Churn.csv is in the data/ folder
df = pd.read_csv(r'C:\Users\USER\Desktop\customer-churn-prediction\data\WA_Fn-UseC_-Telco-Customer-Churn.csv')

print('=== Dataset Shape ===')
print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')

print('\n=== Column Names ===')
print(df.columns.tolist())

print('\n=== Data Types ===')
print(df.dtypes)

df.head()

In [ ]:
print('=== Basic Statistics ===')
df.describe(include='all')

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Churn Distribution ===')
print(df['Churn'].value_counts())
print(f'\nChurn Rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')

---
## STEP 3 — Exploratory Data Analysis (EDA)

In [ ]:
# --- 3.1 Churn Distribution ---
fig, axes = plt.subplots(1, 2)

churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Churn Distribution', fontweight='bold')

sns.countplot(data=df, x='Churn', palette=['#4CAF50', '#F44336'], ax=axes[1])
axes[1].set_title('Churn Count', fontweight='bold')
axes[1].set_xlabel('')

plt.suptitle('Customer Churn Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\churn_distribution.png', dpi=150)
plt.show()

In [ ]:
# --- 3.2 Churn by Contract Type, Internet Service, Payment Method ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['Contract', 'InternetService', 'PaymentMethod']):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x=='Yes').mean() * 100)
    churn_rate.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Churn Rate by {col}', fontweight='bold')
    ax.set_xlabel('Churn Rate (%)')
    ax.set_ylabel('')

plt.suptitle('Churn Rate by Key Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\churn_by_category.png', dpi=150)
plt.show()

In [ ]:
# --- 3.3 Numeric Features Distribution by Churn ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    df_plot = df.copy()
    df_plot[col] = pd.to_numeric(df_plot[col], errors='coerce')
    sns.histplot(data=df_plot, x=col, hue='Churn', kde=True,
                 palette=['#4CAF50', '#F44336'], ax=ax, alpha=0.6)
    ax.set_title(f'{col} by Churn', fontweight='bold')

plt.suptitle('Numeric Feature Distributions by Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\numeric_distributions.png', dpi=150)
plt.show()

In [ ]:
# --- 3.4 Correlation Heatmap ---
df_corr = df.copy()
df_corr['TotalCharges'] = pd.to_numeric(df_corr['TotalCharges'], errors='coerce')
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num']
corr_matrix = df_corr[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Correlation Matrix — Numeric Features', fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\correlation_heatmap.png', dpi=150)
plt.show()

---
## STEP 4 — Data Cleaning & Preprocessing

In [ ]:
# Work on a clean copy
df_clean = df.copy()

# Drop customerID — not a predictive feature
df_clean.drop('customerID', axis=1, inplace=True)

# Fix TotalCharges — it's stored as string, convert to numeric
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with median (only 11 missing — new customers)
df_clean['TotalCharges'].fillna(df_clean['TotalCharges'].median(), inplace=True)

# Encode target variable
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)

# Encode SeniorCitizen is already 0/1 — leave it

# Encode binary Yes/No columns
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df_clean[col] = (df_clean[col] == 'Yes').astype(int)

print('=== After Basic Cleaning ===')
print(f'Shape: {df_clean.shape}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
df_clean.head()

---
## STEP 5 — Feature Engineering

In [ ]:
# --- 5.1 One-Hot Encode remaining categorical columns ---
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode: {cat_cols}')

df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

print(f'\nShape after encoding: {df_encoded.shape}')
print(f'Columns: {df_encoded.columns.tolist()}')

In [ ]:
# --- 5.2 Engineer new features ---

# Average monthly spend relative to tenure
df_encoded['AvgMonthlySpend'] = df_encoded['TotalCharges'] / (df_encoded['tenure'] + 1)

# Tenure group — segment customers by how long they've been with the company
df_encoded['TenureGroup'] = pd.cut(
    df_encoded['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=[1, 2, 3, 4]
).astype(int)

print('✅ New features engineered:')
print('  - AvgMonthlySpend: TotalCharges / (tenure + 1)')
print('  - TenureGroup    : 1=0-12m, 2=12-24m, 3=24-48m, 4=48-72m')
print(f'\nFinal feature count: {df_encoded.shape[1] - 1}')

---
## STEP 6 — Train/Test Split

In [ ]:
# Separate features and target
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Stratified split — preserves churn ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numeric features
scaler = StandardScaler()
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend']
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features]  = scaler.transform(X_test[numeric_features])

print(f'Training set   : {X_train.shape}')
print(f'Test set       : {X_test.shape}')
print(f'Churn rate train: {y_train.mean():.1%}')
print(f'Churn rate test : {y_test.mean():.1%}')

---
## STEP 7 — Logistic Regression (Baseline Model)

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

# Predictions
lr_pred  = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

print('=== Logistic Regression Results ===')
print(f'Accuracy  : {accuracy_score(y_test, lr_pred):.4f}')
print(f'Precision : {precision_score(y_test, lr_pred):.4f}')
print(f'Recall    : {recall_score(y_test, lr_pred):.4f}')
print(f'F1 Score  : {f1_score(y_test, lr_pred):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, lr_proba):.4f}')
print('\n=== Classification Report ===')
print(classification_report(y_test, lr_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion Matrix — Logistic Regression
cm = confusion_matrix(y_test, lr_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix — Logistic Regression', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\confusion_matrix_lr.png', dpi=150)
plt.show()

---
## STEP 8 — Advanced Models (Random Forest & Gradient Boosting)

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest Results ===')
print(f'Accuracy  : {accuracy_score(y_test, rf_pred):.4f}')
print(f'Precision : {precision_score(y_test, rf_pred):.4f}')
print(f'Recall    : {recall_score(y_test, rf_pred):.4f}')
print(f'F1 Score  : {f1_score(y_test, rf_pred):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, rf_proba):.4f}')

In [ ]:
# Train Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

gb_pred  = gb.predict(X_test)
gb_proba = gb.predict_proba(X_test)[:, 1]

print('=== Gradient Boosting Results ===')
print(f'Accuracy  : {accuracy_score(y_test, gb_pred):.4f}')
print(f'Precision : {precision_score(y_test, gb_pred):.4f}')
print(f'Recall    : {recall_score(y_test, gb_pred):.4f}')
print(f'F1 Score  : {f1_score(y_test, gb_pred):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, gb_proba):.4f}')

---
## STEP 9 — Model Comparison & ROC Curves

In [ ]:
# --- 9.1 Model Comparison Table ---
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Accuracy' : [accuracy_score(y_test, lr_pred), accuracy_score(y_test, rf_pred), accuracy_score(y_test, gb_pred)],
    'Precision': [precision_score(y_test, lr_pred), precision_score(y_test, rf_pred), precision_score(y_test, gb_pred)],
    'Recall'   : [recall_score(y_test, lr_pred), recall_score(y_test, rf_pred), recall_score(y_test, gb_pred)],
    'F1 Score' : [f1_score(y_test, lr_pred), f1_score(y_test, rf_pred), f1_score(y_test, gb_pred)],
    'ROC-AUC'  : [roc_auc_score(y_test, lr_proba), roc_auc_score(y_test, rf_proba), roc_auc_score(y_test, gb_proba)]
})

results = results.round(4).set_index('Model')
print('=== Model Comparison ===')
results

In [ ]:
# --- 9.2 ROC Curves ---
plt.figure(figsize=(8, 6))

for name, proba in [
    ('Logistic Regression', lr_proba),
    ('Random Forest',       rf_proba),
    ('Gradient Boosting',   gb_proba)
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1], [0,1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\roc_curves.png', dpi=150)
plt.show()

---
## STEP 10 — Feature Importance Analysis

In [ ]:
# Top 15 most important features from Gradient Boosting (best model)
feature_importance = pd.DataFrame({
    'Feature'  : X_train.columns,
    'Importance': gb.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='Importance', y='Feature', palette='rocket')
plt.title('Top 15 Features Driving Customer Churn', fontweight='bold')
plt.xlabel('Feature Importance Score')
plt.ylabel('')
plt.tight_layout()
plt.savefig(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\feature_importance.png', dpi=150)
plt.show()

print('Top 5 churn drivers:')
print(feature_importance.head())

---
## STEP 11 — Statsmodels Logistic Regression (Statistical Inference)

In [ ]:
# Statsmodels gives us p-values and confidence intervals
# Select top features for interpretability
top_features = feature_importance['Feature'].head(10).tolist()

X_train_sm = sm.add_constant(X_train[top_features])
X_test_sm  = sm.add_constant(X_test[top_features])

logit_model = sm.Logit(y_train, X_train_sm)
logit_result = logit_model.fit(disp=0)

print(logit_result.summary2())

In [ ]:
# Extract significant features (p < 0.05)
summary_df = pd.DataFrame({
    'Coefficient': logit_result.params,
    'P-Value'    : logit_result.pvalues,
    'Odds Ratio' : np.exp(logit_result.params)
})

print('=== Statistically Significant Features (p < 0.05) ===')
print(summary_df[summary_df['P-Value'] < 0.05].round(4))

---
## STEP 12 — Business Insights & Recommendations

In [ ]:
# Generate churn risk scores on full dataset
X_full = df_encoded.drop('Churn', axis=1).copy()
X_full[numeric_features] = scaler.transform(X_full[numeric_features])

df_results = df_encoded.copy()
df_results['ChurnProbability'] = gb.predict_proba(X_full)[:, 1]
df_results['ChurnPrediction']  = gb.predict(X_full)
df_results['RiskSegment'] = pd.cut(
    df_results['ChurnProbability'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print('=== Customer Risk Segmentation ===')
print(df_results['RiskSegment'].value_counts())

print('\n=== High Risk Customer Profile ===')
high_risk = df_results[df_results['RiskSegment'] == 'High Risk']
print(f'Count          : {len(high_risk)}')
print(f'Avg Monthly Charges: £{high_risk["MonthlyCharges"].mean():.2f}')
print(f'Avg Tenure     : {high_risk["tenure"].mean():.1f} months')

In [ ]:
# Risk Segment Distribution Chart
risk_counts = df_results['RiskSegment'].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=risk_counts.index, y=risk_counts.values,
            palette=['#4CAF50', '#FF9800', '#F44336'])
plt.title('Customer Risk Segmentation', fontweight='bold', fontsize=13)
plt.xlabel('Risk Segment')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.savefig(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\risk_segmentation.png', dpi=150)
plt.show()

---
## STEP 13 — Export Predictions & Save Model

In [ ]:
# Export predictions to CSV
output_cols = ['Churn', 'ChurnProbability', 'ChurnPrediction', 'RiskSegment']
df_results[output_cols].to_csv(r'C:\Users\USER\Desktop\customer-churn-prediction\outputs\churn_predictions.csv', index=False)
print('✅ Predictions exported to outputs/churn_predictions.csv')

# Save the best model
joblib.dump(gb, r'C:\Users\USER\Desktop\customer-churn-prediction\models\churn_model_gb.pkl')
joblib.dump(scaler, r'C:\Users\USER\Desktop\customer-churn-prediction\models\scaler.pkl')
print('✅ Model saved to models/churn_model_gb.pkl')
print('✅ Scaler saved to models/scaler.pkl')

---
## Summary

| Model | Accuracy | ROC-AUC |
|---|---|---|
| Logistic Regression | ~80% | ~0.85 |
| Random Forest | ~79% | ~0.83 |
| Gradient Boosting | ~80% | ~0.85 |

### Key Findings
- **Tenure** is the strongest churn predictor — new customers churn most
- **Month-to-month contracts** have significantly higher churn rates than annual contracts
- **High monthly charges** correlate strongly with churn risk
- **Fiber optic internet** customers churn more than DSL customers

### Business Recommendations
1. **Target new customers (0–12 months)** with onboarding incentives and loyalty discounts
2. **Offer annual contract upgrades** to month-to-month customers with high churn probability
3. **Review fiber optic pricing** — high charges + high churn = pricing friction
4. **Deploy this model in production** to score all customers monthly and trigger retention campaigns for High Risk segment